In [ ]:
import pandas as pd
from typing import List, Dict, Optional, Any
import re
from pydantic import BaseModel, Field, ConfigDict


class DatasetMetadata(BaseModel):
    name: str
    path: str
    profile_summary: Optional[Dict[str, Any]] = None
    schema: Optional[List[str]] = None

class SchemaMatch(BaseModel):
    dataset_a: str
    dataset_b: str
    matches: Optional[List[Dict[str, Any]]] = None
    generated_code: Optional[str] = None

class Issue(BaseModel):
    description: str
    severity: str = "medium"
    agent: Optional[str] = None
    resolved: bool = False

class IntegrationState(BaseModel):
    """
    Central state shared across all agents.
    Tracks datasets, schema matches, issues, and final fused output.
    """
    # FIX: Allow Pydantic to handle the complex pd.DataFrame type
    model_config = ConfigDict(arbitrary_types_allowed=True)

    datasets: List[DatasetMetadata] = Field(default_factory=list)
    schema_matches: List[SchemaMatch] = Field(default_factory=list)
    issues: List[Issue] = Field(default_factory=list)
    fused_data: Optional[pd.DataFrame] = None

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "DatasetMetadata" shadows an attribute in parent "BaseModel"
  warnings.warn(


In [5]:
from PyDI.profiling import DataProfiler
from PyDI.io import load_csv, load_xml, load_parquet
# from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

# --- UTILITY FUNCTION ---
def extract_python_code(response_text: str) -> Optional[str]:
    """Extracts the Python code block from the LLM response."""
    # Regex to find content inside triple backticks with optional 'python' marker
    match = re.search(r"```python\s*(.*?)\s*```", response_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None

# --- AGENT TOOLS ---

# TOOL 1: Dataset Loader + Profiler
@tool
def load_and_profile_dataset(file_path: str) -> str:
    """
    Loads dataset (CSV, Parquet, XML) and profiles it using PyDI DataProfiler.
    The output is a JSON string summarizing the dataset's profile and a 3-row sample.
    """
    ext = os.path.splitext(file_path)[1].lower()
    df = None
    try:
        if ext == ".csv":
            df = load_csv(file_path)
        elif ext == ".parquet":
            df = load_parquet(file_path)
        elif ext == ".xml":
            df = load_xml(file_path)
        else:
            raise ValueError(f"Unsupported format: {ext}. Supported: CSV, Parquet, XML.")

        if df is None or df.empty:
             return f"Error: Dataset at {file_path} loaded as empty or failed to load."

        profiler = DataProfiler()
        profile = profiler.summary(df)

        return json.dumps({
            "file_path": file_path,
            "profile": profile,
            "sample": df.head(3).to_dict(orient="records")
        }, indent=2)
    except Exception as e:
        return f"Error loading or profiling dataset at {file_path}: {e}"

# TOOL 2: PyDI Docs Retriever
@tool
def get_pydi_doc(module_name: str) -> str:
    """
    Fetch documentation for a PyDI module or class.
    Example usage: get_pydi_doc('PyDI.schemamatching.SchemaMatcher')
    """
    try:
        components = module_name.split(".")
        module = __import__(components[0])
        for comp in components[1:]:
            module = getattr(module, comp)
        return module.__doc__ or f"No docstring available for {module_name}."
    except Exception as e:
        return f"Error retrieving docs for {module_name}: {e}"

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.2,
    max_tokens=512
)

integration_tools = [load_and_profile_dataset, get_pydi_doc]

# 1. SCHEMA INTEGRATION AGENT (Code Generator)

# integration_agent_prompt = hub.pull("hwchase17/structured-chat-agent")

# Define a custom prompt suitable for create_tool_calling_agent
integration_agent_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI assistant with access to tools."),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

integration_agent_runnable = create_tool_calling_agent(llm, integration_tools, integration_agent_prompt)

integration_agent_executor = AgentExecutor(
    agent=integration_agent_runnable,
    tools=integration_tools,
    verbose=True,
    handle_parsing_errors=True
)

# 2. CODE VALIDATOR AGENT (Code Debugger)
# This agent does not need tools; its job is pure reasoning/critique.
validator_prompt_template = """
You are an expert Python Code Validator and Debugger.

Your current task is to review and fix the PyDI Python script below.
Assume the code was run and failed with a generic error (e.g., missing import, incorrect function call, syntax mistake).

The code generated is:
--- CODE START ---
{generated_code}
--- CODE END ---

**Self-Healing/Fixing Instructions:**
1.  **Critique:** Review the code for all common errors (syntax, logic, missing imports, PyDI method signatures).
2.  **Fix:** If an error is found, rewrite the entire, corrected, fully working Python script.
3.  **Validation:** If the code appears perfect and ready to run, respond **ONLY** with the phrase "CODE_OK".

Respond ONLY with the complete, corrected Python code inside triple backticks (```python ... ```) or the exact phrase "CODE_OK". Do not include any other text or reasoning.
"""
validator_prompt = ChatPromptTemplate.from_template(validator_prompt_template)
validator_chain = validator_prompt | llm | StrOutputParser()

In [14]:
# 3. IDENTITY RESOLUTION AGENT (Blocking & Matching Code Generator)
resolution_prompt_template = """
You are an expert PyDI Entity Resolution Agent specializing in blocking and matching.

A previous agent successfully generated and validated the **Schema Matching** code.
Your goal is to complete the pipeline by generating the code for **Identity Resolution**.

**Context (Validated Schema Matching Code):**
--- PREVIOUS CODE START ---
{validated_schema_match_code}
--- PREVIOUS CODE END ---

**Your Task:**
1.  **Assume** the code in the 'PREVIOUS CODE' block runs successfully and variables like `df_a`, `df_b`, and the schema mapping result are available.
2.  **Use your PyDI knowledge and the 'get_pydi_doc' tool** to determine the optimal blocking and matching methods (e.g., `StandardBlocker`, `SortedNeighbourhoodBlocker`, `TokenBlocker`) based on the assumed schema.
3.  **Generate a complete, fully executable PyDI Python script** that performs the *entire* workflow:
    * Imports (from PyDI.io, PyDI.profiling, PyDI.entitymatching, etc.)
    * Dataset Loading (using the provided paths)
    * **Schema Matching (Integrate the validated code)**
    * **Blocking (Blocking setup and execution)**
    * **Matching/Pairing (Matcher setup and execution)**

Respond ONLY with the complete, end-to-end Python script inside triple backticks.
"""
resolution_agent_runnable = create_tool_calling_agent(llm, integration_tools, integration_agent_prompt.partial(
    system=resolution_prompt_template
))
resolution_agent_executor = AgentExecutor(
    agent=resolution_agent_runnable,
    tools=integration_tools,
    verbose=True,
    handle_parsing_errors=True
)

In [15]:
# REUSABLE VALIDATION FUNCTION
def validate_code_block(code_to_validate: str, step_name: str) -> str:
    """Helper function to run code through the validation agent."""
    print(f"\n--- Running Validator for {step_name} ---")

    validation_input = {"generated_code": code_to_validate}
    validation_result = validator_chain.invoke(validation_input)

    if validation_result.strip() == "CODE_OK":
        print(f"[+] Validation SUCCESS: {step_name} code confirmed as correct.")
        return code_to_validate
    else:
        fixed_code = extract_python_code(validation_result)
        if fixed_code:
            print(f"[+] Validation FIX: {step_name} code was automatically debugged and rewritten.")
            return fixed_code
        else:
            print(f"[!] Validation FAILED: Validator output for {step_name} was not clean code. Using initial code.")
            return code_to_validate

In [16]:
TASK_INPUT = {
    "dataset_a_path": "datasets/amazon.patquet",
    "dataset_b_path": "datasets/goodreads.patquet",
    "dataset_c_path": "datasets/metabooks.patquet"
}

# Define the task prompt for the Integration Agent
integration_task_description = f"""
You are an expert Autonomous Schema Integration Agent specializing in data
manipulation and integration using the PyDI library.

**1. Primary Goal:**
Your overarching task is to produce a single, minimal, and fully functional
**PyDI Python script** that performs schema matching between the provided datasets.

**2. Execution Steps (Follow this order):**
1.  **Analyze Inputs:** Use `load_and_profile_dataset` on **Dataset A**, **Dataset B**, and **Dataset C**
    to understand their structure, columns, and data types.
2.  **Research:** Use `get_pydi_doc` to look up necessary PyDI modules, particularly
    for schema matching (e.g., PyDI.schemamatching or PyDI.entitymatching).
3.  **Code Generation:** Based on the dataset profiles and PyDI documentation,
    write the complete Python code for schema matching.

**3. Datasets to Use:**
* Dataset A Path: {TASK_INPUT['dataset_a_path']}
* Dataset B Path: {TASK_INPUT['dataset_b_path']}
* Dataset C Path: {TASK_INPUT['dataset_c_path']}

**4. Final Output Format:**
* Your final, complete response MUST contain ONLY the Python code
    required to load the datasets, define the schema matching process using PyDI,
    and execute the matching.
* The code must be enclosed **ONLY within a single set of triple backticks** (```python ... ```).
* Do NOT include any explanatory text, commentary, or conversational dialogue outside of the code block.

Begin by calling your first tool to load and profile '{TASK_INPUT['dataset_a_path']}'.
"""


In [17]:
print("--- 1. Starting Schema Integration Agent (Code Generation) ---")
integration_response = integration_agent_executor.invoke({
    "input": integration_task_description
})

--- 1. Starting Schema Integration Agent (Code Generation) ---


> Entering new AgentExecutor chain...


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
initial_code = extract_python_code(integration_response["output"])
print("\n[+] Initial Code Generated (Excerpt):\n", initial_code[:200] + "..." if initial_code else "None")

if not initial_code:
    print("\n[!] FATAL ERROR: Integration Agent failed to generate code.")

# 2. Run Code Validator Agent (Validation for Schema Match Code)
validated_schema_code = validate_code_block(initial_code, "Schema Matching Code")

In [ ]:
# 3. Run Identity Resolution Agent (Blocking & Matching Code Generation)
print("\n--- 3. Starting Identity Resolution Agent (Blocking & Matching Code Generation) ---")

resolution_prompt_context = resolution_prompt_template.format(
    validated_schema_match_code=validated_schema_code
)

resolution_response = resolution_agent_executor.invoke({
    "input": resolution_prompt_context
})
resolution_code = extract_python_code(resolution_response["output"])

if not resolution_code:
    print("\n[!] FATAL ERROR: Resolution Agent failed to generate code.")

# 4. Run Code Validator Agent (Validation for Identity Resolution Code)
final_pipeline_code = validate_code_block(resolution_code, "Identity Resolution Pipeline")

print(final_pipeline_code)